In [32]:
# prosty model do oceny czy wiadomosc jest spamem czy nie
# dzialanie modelu:
# bazuje na prawdopobobienswie warunkowym, 
# ocenia na podstawie bazy danych jakie jest prawd. ze pojedyncze slowo znalazlo sie w wiadomosci spam 
# i zaklada ze slowa sa NIEZALEZNE

In [33]:
from typing import Set, NamedTuple, List, Tuple, Dict, Iterable, TypeVar
import math
import re
from collections import defaultdict

def tokenize(text: str) -> Set[str]:
    'towrzy set z unikalnymi slowami w wiadomosci'
    text = text.lower()
    all_words = re.findall("[a-z0-9]+", text)
    return set(all_words)

assert tokenize("Data science is science") == {'data', 'is', 'science'}

# sposob na przechowywanie danych
class Message(NamedTuple):
    text: str
    is_spam: bool

class NaiveBayesClassifier:
    def __init__(self, k: float = 0.5) -> None:
        self.k = k
        self.tokens: Set[str] = set()
        self.token_spam_counts: Dict[str, int] = defaultdict(int)
        self.token_ham_counts: Dict[str, int] = defaultdict(int)
        self.spam_messages = self.ham_messages = 0

    def train(self, messages : Iterable[Message]) -> None:
        for message in messages:
            if message.is_spam:
                self.spam_messages += 1
            else:
                self.ham_messages += 1
            for token in tokenize(message.text):
                self.tokens.add(token)
                if message.is_spam:
                    self.token_spam_counts[token] += 1 
                else:
                    self.token_ham_counts[token] += 1 
    
    def _probabilities(self, token:str) -> Tuple[float, float]:
        spam = self.token_spam_counts[token]
        ham = self.token_ham_counts[token]
        p_token_spam = (spam + self.k) / (self.spam_messages + 2* self.k)
        p_token_ham = (ham + self.k) / (self.ham_messages + 2* self.k)
    
        return p_token_spam, p_token_ham
    
    def predict(self, text: str) -> float:
        text_tokens = tokenize(text)
        log_prob_is_spam = log_prob_is_ham = 0.0
        for token in self.tokens:
            prob_is_spam, prob_is_ham = self._probabilities(token)

            if token in text_tokens:
                log_prob_is_spam += math.log(prob_is_spam)
                log_prob_is_ham += math.log(prob_is_ham)
            
            else:
                log_prob_is_spam += math.log(1.0 - prob_is_spam)
                log_prob_is_ham += math.log(1.0 - prob_is_ham)
        prob_if_spam = math.exp(log_prob_is_spam)
        prob_if_ham = math.exp(log_prob_is_ham)

        return prob_if_spam / (prob_if_spam + prob_if_ham)

In [34]:
# testowanie modelu 
messages = [Message("spam rules", is_spam=True),
            Message("ham rules", is_spam=False),
            Message("hello ham", is_spam=False)]

model = NaiveBayesClassifier(k=0.5)
model.train(messages)

assert model.tokens == {"spam", "ham", "rules", "hello"}
assert model.spam_messages == 1
assert model.ham_messages == 2
assert model.token_spam_counts == {"spam": 1, "rules": 1}
assert model.token_ham_counts == {"ham": 2, "rules": 1, "hello": 1}

In [35]:
from io import BytesIO  # So we can treat bytes as a file.
import requests         # To download the files, which
import tarfile          # are in .tar.bz format.

BASE_URL = "https://spamassassin.apache.org/old/publiccorpus"
FILES = ["20021010_easy_ham.tar.bz2",
         "20021010_hard_ham.tar.bz2",
         "20021010_spam.tar.bz2"]

# This is where the data will end up,
# in /spam, /easy_ham, and /hard_ham subdirectories.
# Change this to where you want the data.
OUTPUT_DIR = 'spam_data'

for filename in FILES:
    # Use requests to get the file contents at each URL.
    content = requests.get(f"{BASE_URL}/{filename}").content

    # Wrap the in-memory bytes so we can use them as a "file."
    fin = BytesIO(content)

    # And extract all the files to the specified output dir.
    with tarfile.open(fileobj=fin, mode='r:bz2') as tf:
        tf.extractall(OUTPUT_DIR)

/var/folders/02/58ywv9f514l59fv2ts8pdcn80000gn/T/ipykernel_92257/1239019972.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(OUTPUT_DIR)


In [36]:
import glob, re

path = '/Users/ignacyboczula/Desktop/Data_science_Joel_Grus/chapter_13/spam_data/*/*'
data: List[Message] = []

for filename in glob.glob(path):
    is_spam = "ham" not in filename

    with open(filename, errors='ignore') as email_file:
        for line in email_file:
            if line.startswith("Subject:"):
                subject = line.lstrip("Subject: ")
                data.append(Message(subject, is_spam))
                break  

import random
X = TypeVar('X')
def split_data(data: List[X], prob: float) -> Tuple[List[X], List[X]]:
    'dzieli dane na dwie czesci (prob, 1-prob)'
    data = data[:]
    random.shuffle(data)
    cut = int(len(data) * prob)
    return data[:cut], data[cut:]

random.seed(0)      
train_messages, test_messages = split_data(data, 0.75)

model = NaiveBayesClassifier()
model.train(train_messages)

from collections import Counter

predictions = [(message, model.predict(message.text))
               for message in test_messages]


confusion_matrix = Counter((message.is_spam, spam_probability > 0.5)
                           for message, spam_probability in predictions)

print(confusion_matrix)

Counter({(False, False): 664, (True, True): 85, (True, False): 54, (False, True): 22})


In [37]:
def p_spam_given_token(token: str, model: NaiveBayesClassifier) -> float:
    # We probably shouldn't call private methods, but it's for a good cause.
    prob_if_spam, prob_if_ham = model._probabilities(token)

    return prob_if_spam / (prob_if_spam + prob_if_ham)

words = sorted(model.tokens, key=lambda t: p_spam_given_token(t, model))

print("spammiest_words", words[-10:])
print("hammiest_words", words[:10])

spammiest_words ['per', '500', 'attn', 'sale', 'account', 'zzzz', 'systemworks', 'money', 'adv', 'rates']
hammiest_words ['spambayes', '2', 'users', 'razor', 'zzzzteana', 'sadev', 'ouch', 'apt', 'bliss', 'selling']
